# Fetching country pres/abs for all species from GLOBI dataset

In [3]:
# ==============================================================================
# GBIF PRESENCE ENRICHMENT — FULL GRAPH
#
# Fetches yes/no country presence for every resolvable species in the graph
# (all SINAS invasives + all GloBI interaction partners), then builds:
#   - presence      : (num_species, num_countries) bool tensor
#   - bio_opp_mat   : (num_species, num_countries) float16 tensor
#                     bio_opp_mat[sp, co] = fraction of sp's GloBI partners
#                     confirmed present in co — used as biotic opportunity score
#
# Run once overnight on Colab (Google-to-GBIF latency ~5ms vs ~80ms local).
# Saves to Google Drive so it survives session termination.
# Resumes automatically from checkpoint if interrupted.
#
# !pip install aiohttp nest_asyncio country_converter tqdm -q
# ==============================================================================

import asyncio, json, logging, random, time
from collections import defaultdict
from pathlib import Path

import aiohttp
import pandas as pd
import torch
from tqdm.auto import tqdm
import country_converter as coco
import nest_asyncio

logging.getLogger('country_converter').setLevel(logging.ERROR)

c:\Users\simon\Documents\GitHub\horizon-scanner\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [22]:
# ==============================================================================
# CONFIG — adjust these to taste
# ==============================================================================
repo_root   = Path.cwd().parent # get repo root

sinas_path        = repo_root / 'data' / 'sinas_matched_species.csv'
network_path      = repo_root / 'data' / 'matched_globi_network.csv'
higher_order_path = repo_root / 'data' / 'matched_globi_network_higher_order.csv'
output_path       = repo_root / 'data' / 'gbif_presence.pt'
checkpoint_path   = repo_root / 'data' / 'gbif_presence_checkpoint.json'  # resume support

YOUR_EMAIL      = 'simon.reynaert@plantentuinmeise.be'  # ← tells GBIF who you are; gets better rate limits
MAX_CONCURRENT  = 50     # 10 workers → ~10 sp/s steady, no 429s
REQUEST_TIMEOUT = 30     # seconds per request before timeout
FACET_LIMIT     = 300    # covers all ~250 GBIF country codes
MAX_RETRIES     = 3      # per-request retry attempts
CHECKPOINT_EVERY = 200   # write checkpoint every N species
country_col     = 'location'


In [5]:

# ==============================================================================
# 1. REBUILD INDEX MAPPINGS
# ==============================================================================
print("📖 Rebuilding index mappings...")

df_sinas  = pd.read_csv(sinas_path,        engine='python', on_bad_lines='warn')
df_net    = pd.read_csv(network_path,       engine='python', on_bad_lines='warn')
df_higher = pd.read_csv(higher_order_path,  engine='python', on_bad_lines='warn')

df_sinas = df_sinas.dropna(subset=['gbif_id']).copy()
df_sinas['gbif_id'] = df_sinas['gbif_id'].astype(int)

sinas_sp   = set(df_sinas['gbif_id'].unique())
net_src_sp = set(df_net['source_gbif_id'].dropna().astype(int).unique())
net_tgt_sp = set(df_net['target_gbif_id'].dropna().astype(int).unique())
all_species_ids   = sorted(sinas_sp | net_src_sp | net_tgt_sp)
all_country_names = sorted(df_sinas[country_col].dropna().unique().tolist())

sp_to_idx = {sp_id: idx for idx, sp_id in enumerate(all_species_ids)}
co_to_idx = {name:  idx for idx, name  in enumerate(all_country_names)}

num_species   = len(all_species_ids)
num_countries = len(all_country_names)

# Build id_to_name for resolvability check
id_to_name = (df_sinas.drop_duplicates('gbif_id')
                       .set_index('gbif_id')['gbif_canonical_name']
                       .to_dict())
for row in df_net.dropna(subset=['source_gbif_id']).drop_duplicates('source_gbif_id').itertuples():
    if int(row.source_gbif_id) not in id_to_name:
        id_to_name[int(row.source_gbif_id)] = row.source_taxon_name
for row in df_net.dropna(subset=['target_gbif_id']).drop_duplicates('target_gbif_id').itertuples():
    if int(row.target_gbif_id) not in id_to_name:
        id_to_name[int(row.target_gbif_id)] = row.target_taxon_name

# Resolvable = has a real GBIF backbone name (not a placeholder ID)
def is_resolvable(gbif_id):
    name = str(id_to_name.get(gbif_id, ''))
    return not any(name.startswith(p) for p in ('GBIF:', 'Unresolved', 'nan'))

resolvable = {
    gid: sp_to_idx[gid]
    for gid in all_species_ids
    if is_resolvable(gid)
}

print(f"   Total species in graph : {num_species:,}")
print(f"   Resolvable (will fetch): {len(resolvable):,}")
print(f"   Unresolvable (skipped) : {num_species - len(resolvable):,}  "
      f"← placeholder GBIF: IDs from unmatched GloBI records")
print(f"   Countries              : {num_countries:,}")


📖 Rebuilding index mappings...
   Total species in graph : 167,357
   Resolvable (will fetch): 167,357
   Unresolvable (skipped) : 0  ← placeholder GBIF: IDs from unmatched GloBI records
   Countries              : 289


In [6]:
# ==============================================================================
# 2. ISO2 → COUNTRY INDEX MAPPING
# ==============================================================================
SUB_NATIONAL_TO_ISO2 = {
    'Aegean':'GR','Caspian Sea':'RU','Alaska':'US','Hawaii':'US',
    'Virgin Islands (U.S.)':'US','United States Minor Outlying Islands':'US',
    'Puerto Rico':'US','Guam':'GU','American Samoa':'AS',
    'Northern Mariana Islands':'MP','Canary Islands':'ES','Balearic Islands':'ES',
    'Azores':'PT','Madeira':'PT','Sicily':'IT','Sardinia':'IT',
    'Tasmania':'AU','Lord Howe Islands':'AU','Norfolk Island':'NF',
    'Christmas Island':'CX','Cocos Islands':'CC','Galapagos':'EC',
    'Corsica':'FR','Réunion':'RE','Mayotte':'YT','Guadeloupe':'GP',
    'Martinique':'MQ','French Guiana':'GF','Saint Pierre and Miquelon':'PM',
    'Saint-Barthélemy':'BL','Saint-Martin':'MF','Shetland Islands':'GB',
    'Guernsey':'GG','Jersey':'JE','Isle of Man':'IM','Gibraltar':'GI',
    'Bermuda':'BM','Anguilla':'AI','Cayman Islands':'KY','Montserrat':'MS',
    'Turks and Caicos Islands':'TC','Virgin Islands (British)':'VG',
    'Falkland Islands':'FK','Pitcairn Islands':'PN','Saint Helena':'SH',
    'Vancouver Island':'CA','Crete':'GR','Zanzibar Island':'TZ',
    'Rapa Nui':'CL','Socotra Island':'YE','Nicobar and Andaman Islands':'IN',
    'Rodriguez Island':'MU','Hong Kong':'HK','Macao':'MO',
    'Faroe Islands':'FO','Greenland':'GL','Svalbard and Jan Mayen':'SJ',
    'Åland':'AX','Tokelau':'TK','Niue':'NU','Cook Islands':'CK',
    'Kermadec Islands':'NZ','Northern Cyprus':'CY','Western Sahara':'EH',
    'Fernando de Noronha':'BR','Palestine':'PS','Kosovo':'XK',
    'Antipodes Island':'NZ','Izu Islands':'JP','Ogasawara Islands':'JP',
}

cc = coco.CountryConverter()
iso2_to_co_indices = defaultdict(list)
for name in all_country_names:
    co_idx = co_to_idx[name]
    iso2   = SUB_NATIONAL_TO_ISO2.get(name) or cc.convert(names=name, to='ISO2')
    if iso2 and iso2 != 'not_found':
        iso2_to_co_indices[iso2.upper()].append(co_idx)

print(f"   ISO2 codes mapped: {len(iso2_to_co_indices):,}")


   ISO2 codes mapped: 246


In [7]:
# ==============================================================================
# 3. SEED PRESENCE MATRIX FROM SINAS
# ==============================================================================
presence = torch.zeros(num_species, num_countries, dtype=torch.bool)

df_sinas['sp_idx'] = df_sinas['gbif_id'].map(sp_to_idx)
df_sinas['co_idx'] = df_sinas[country_col].map(co_to_idx)
seed_df = df_sinas.dropna(subset=['sp_idx', 'co_idx'])
presence[seed_df['sp_idx'].astype(int).values,
         seed_df['co_idx'].astype(int).values] = True
n_seeded = presence.sum().item()
print(f"   Seeded {n_seeded:,} presences from SINAS")



   Seeded 356,506 presences from SINAS


In [33]:
# ==============================================================================
# 4. LOAD CHECKPOINT (resume support)
# ==============================================================================
completed  = {}
failed_ids = []

if checkpoint_path.exists():
    try:
        with open(checkpoint_path) as f:
            ckpt = json.load(f)
        completed  = {int(k): v for k, v in ckpt.get('completed', {}).items()}
        failed_ids = ckpt.get('failed', [])

        # Replay already-fetched data into presence matrix
        for gbif_id, co_indices in completed.items():
            sp_idx = resolvable.get(gbif_id)
            if sp_idx is not None and co_indices:
                presence[sp_idx,
                         torch.tensor(co_indices, dtype=torch.long)] = True

        print(f"\n♻️  Resuming from checkpoint:")
        print(f"   Already completed: {len(completed):,} species")
        print(f"   Previously failed: {len(failed_ids):,} species")
    except json.JSONDecodeError:
        print("⚠️  Corrupt checkpoint (interrupted write) — starting fresh.")
        checkpoint_path.unlink()

remaining = {
    gid: sidx
    for gid, sidx in resolvable.items()
    if gid not in completed
}
print(f"   Remaining to fetch: {len(remaining):,} species")


♻️  Resuming from checkpoint:
   Already completed: 167,357 species
   Previously failed: 251,638 species
   Remaining to fetch: 0 species


In [9]:
# ==============================================================================
# 5. ASYNC FETCHER
# ==============================================================================
# limit=0 handles occurrence queries instantly by requesting counts only
GBIF_FACET_URL = (
    "https://api.gbif.org/v1/occurrence/search"
    "?taxonKey={taxon_key}"
    "&limit=0"
    "&facet=country"
    f"&facetLimit={FACET_LIMIT}"
)

async def fetch_one(session, gbif_id, semaphore):
    url = GBIF_FACET_URL.format(taxon_key=gbif_id)
    for attempt in range(MAX_RETRIES):
        try:
            async with semaphore:
                async with session.get(
                    url,
                    timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
                ) as resp:
                    if resp.status == 429:
                        wait = 2 ** attempt * 10 + random.uniform(1, 5)
                        await asyncio.sleep(wait)
                        continue
                    if resp.status != 200:
                        await asyncio.sleep(2.0 * (attempt + 1) + random.uniform(0, 1))
                        continue
                    data = await resp.json(content_type=None)

            co_indices = []
            facets = data.get('facets', [])
            if facets:
                for entry in facets[0].get('counts', []):
                    iso2  = entry.get('name', '').upper()
                    count = entry.get('count', 0)
                    if count > 0 and iso2 in iso2_to_co_indices:
                        co_indices.extend(iso2_to_co_indices[iso2])
            return gbif_id, list(set(co_indices))

        except asyncio.TimeoutError:
            await asyncio.sleep(2.0 * (attempt + 1) + random.uniform(0, 1))
        except Exception:
            await asyncio.sleep(1.0 * (attempt + 1) + random.uniform(0, 1))

    return gbif_id, None


async def run_enrichment(fetch_dict):
    gbif_ids  = list(fetch_dict.keys())
    total     = len(gbif_ids)
    in_queue  = asyncio.Queue()
    out_queue = asyncio.Queue()

    for gid in gbif_ids:
        await in_queue.put(gid)

    semaphore     = asyncio.Semaphore(MAX_CONCURRENT)
    new_completed = {}
    new_failed    = []

    headers = {
        'User-Agent': f'HorizonScanner/1.0 (mailto:{YOUR_EMAIL})',
    }
    connector = aiohttp.TCPConnector(
        limit=MAX_CONCURRENT + 4,
        ttl_dns_cache=300,
        enable_cleanup_closed=True,
    )

    async def worker():
        while True:
            try:
                gid = in_queue.get_nowait()
            except asyncio.QueueEmpty:
                return
            result = await fetch_one(session, gid, semaphore)
            await out_queue.put(result)
            in_queue.task_done()

    async with aiohttp.ClientSession(connector=connector, headers=headers) as session:
        workers = [asyncio.create_task(worker()) for _ in range(MAX_CONCURRENT)]
        processed = 0
        pbar = tqdm(total=total, desc="GBIF fetch", unit="sp", dynamic_ncols=True)
        
        try:
            while processed < total:
                gbif_id, co_indices = await out_queue.get()
                processed += 1

                if co_indices is None:
                    new_failed.append(int(gbif_id))
                else:
                    new_completed[int(gbif_id)] = [int(x) for x in co_indices]
                    sp_idx = fetch_dict[gbif_id]
                    if co_indices:
                        presence[sp_idx, torch.tensor(co_indices, dtype=torch.long)] = True

                pbar.update(1)
                pbar.set_postfix({
                    'ok':     len(new_completed),
                    'failed': len(new_failed),
                }, refresh=False)

                # Windows-hardened Atomic Checkpoint
                if processed % CHECKPOINT_EVERY == 0 or processed == total:
                    all_completed = {
                        int(k): [int(x) for x in v]
                        for k, v in {**completed, **new_completed}.items()
                    }
                    all_failed = [int(x) for x in failed_ids + new_failed]
                    tmp_path = checkpoint_path.with_suffix('.tmp')
                    
                    try:
                        with open(tmp_path, 'w') as f:
                            json.dump({'completed': all_completed, 'failed': all_failed}, f)
                        
                        # Defend against aggressive OS/OneDrive locks
                        for lock_attempt in range(5):
                            try:
                                if checkpoint_path.exists():
                                    checkpoint_path.unlink()
                                tmp_path.rename(checkpoint_path)
                                break
                            except PermissionError:
                                if lock_attempt == 4:
                                    print(f"\n⚠️ Checkpoint write locked by Windows. Skipping this flush, but keeping data in memory...")
                                else:
                                    await asyncio.sleep(0.5)
                    except Exception as ce:
                        print(f"\n⚠️ Error updating checkpoint: {ce}")

        except (asyncio.CancelledError, KeyboardInterrupt):
            print("\n🛑 Fetching interrupted! Halting workers cleanly...")
            raise
        finally:
            # Fixes stuck tickers: force the widget to drop structural connections
            pbar.close()
            
            # Fixes zombie background loops: kill active connections instantly
            for w in workers:
                w.cancel()
            await asyncio.gather(*workers, return_exceptions=True)
            print("🧹 Active connections evicted. Loop is stable.")

    return new_completed, new_failed

In [29]:
# ==============================================================================
# 6. RUN MAIN FETCH
# ==============================================================================
nest_asyncio.apply()
loop = asyncio.get_event_loop()

if remaining:
    eta_min = len(remaining) // max(MAX_CONCURRENT * 6, 1)
    print(f"\n🌐 Fetching {len(remaining):,} species ({MAX_CONCURRENT} workers)...")

    new_completed, new_failed = loop.run_until_complete(run_enrichment(remaining))
    print(f"\n   Fetched : {len(new_completed):,}")
    print(f"   Failed  : {len(new_failed):,}")
else:
    print("\n✅ All species already accounted for — processing from database.")
    new_completed = {}
    new_failed    = []


🌐 Fetching 97 species (50 workers)...


GBIF fetch:   0%|          | 0/97 [00:00<?, ?sp/s]

🧹 Active connections evicted. Loop is stable.

   Fetched : 97
   Failed  : 0


In [34]:

# ==============================================================================
# 7. AUTOMATIC RETRY PASSES FOR FAILED SPECIES
#    Up to 3 passes with increasing backoff. Most failures are transient
#    (timeout spike, momentary 503) and recover on the first retry pass.
# ==============================================================================
# Remove any IDs that eventually succeeded — failed_ids from the checkpoint
# may include species that were retried and recovered in a previous session.
# We only want species genuinely unresolved after the main fetch completes.
all_succeeded = set(completed.keys()) | set(int(k) for k in new_completed.keys())
all_failed    = [int(x) for x in failed_ids + new_failed
                 if int(x) not in all_succeeded]
 
if all_failed:
    print(f"\n🔁 Retrying {len(all_failed):,} failed species "
          f"(up to 3 passes with backoff)...")
 
    for retry_pass in range(1, 4):
        if not all_failed:
            break
 
        backoff = 30 * retry_pass
        print(f"\n   Pass {retry_pass}/3 — waiting {backoff}s then retrying "
              f"{len(all_failed):,} species...")
        time.sleep(backoff)
 
        retry_dict = {
            gid: resolvable[gid]
            for gid in all_failed
            if gid in resolvable
        }
        retry_completed, still_failed = loop.run_until_complete(
            run_enrichment(retry_dict)
        )
 
        new_completed.update(retry_completed)
        for gbif_id, co_indices in retry_completed.items():
            sp_idx = resolvable.get(gbif_id)
            if sp_idx is not None and co_indices:
                presence[sp_idx,
                         torch.tensor(co_indices, dtype=torch.long)] = True
 
        print(f"   Pass {retry_pass} recovered : {len(retry_completed):,} | "
              f"Still failing: {len(still_failed):,}")
        all_failed = [int(x) for x in still_failed]
 
    if all_failed:
        print(f"\n   ⚠️  {len(all_failed):,} species failed all retry passes.")
        print(f"   These will have SINAS-only presence data (or zeros if GloBI-only).")
    else:
        print(f"\n   ✅ All failed species recovered.")
else:
    all_failed = []

In [38]:
 
# ==============================================================================
# 8. BIOTIC OPPORTUNITY MATRIX
#    bio_opp_mat[sp, co] = fraction of sp's GloBI partners present in co.
#    Precomputed here so training-time lookup is a single tensor index op.
#
#    If the output file already exists (from a previous run), we load
#    bio_opp_mat directly from it rather than recomputing — this saves
#    significant time when the script is resumed after multi-day fetching.
# ==============================================================================
if output_path.exists():
    print("\n⚙️ Loading bio_opp_mat from existing output file (skipping recompute)...")
    _existing    = torch.load(output_path, map_location='cpu',weights_only=False)
    bio_opp_mat  = _existing['bio_opp_mat']   # (S, C) float16
    del _existing
    print(f"   Loaded bio_opp_mat: {list(bio_opp_mat.shape)}")
else:
    print("\n⚙️ Precomputing biotic opportunity matrix...")
 
    # Build sp_idx-keyed adjacency from GloBI edges (bidirectional)
    interaction_partners = defaultdict(set)
    net_clean = df_net.dropna(subset=['source_gbif_id', 'target_gbif_id']).copy()
    net_clean['source_gbif_id'] = net_clean['source_gbif_id'].astype(int)
    net_clean['target_gbif_id'] = net_clean['target_gbif_id'].astype(int)
    for row in net_clean.itertuples():
        src_idx = sp_to_idx.get(row.source_gbif_id)
        tgt_idx = sp_to_idx.get(row.target_gbif_id)
        if src_idx is not None and tgt_idx is not None:
            interaction_partners[src_idx].add(tgt_idx)
            interaction_partners[tgt_idx].add(src_idx)
 
    print(f"   Species with known interactions: {len(interaction_partners):,}")
 
    # For each species, mean presence of its partners across all countries.
    # float16 is sufficient precision for a [0,1] fraction and halves memory.
    bio_opp_mat = torch.zeros(num_species, num_countries, dtype=torch.float16)
    presence_f  = presence.float()   # temporary float32 view for mean computation
 
    CHUNK = 500
    species_with_partners = list(interaction_partners.keys())
    for i in tqdm(range(0, len(species_with_partners), CHUNK),
                  desc="Biotic opportunity"):
        for sp in species_with_partners[i: i + CHUNK]:
            partners = torch.tensor(list(interaction_partners[sp]), dtype=torch.long)
            bio_opp_mat[sp] = presence_f[partners].mean(dim=0).half()
 
    del presence_f   # free the temporary float32 copy


⚙️ Loading bio_opp_mat from existing output file (skipping recompute)...
   Loaded bio_opp_mat: [167357, 289]


In [39]:
# ==============================================================================
# 9. SAVE
# ==============================================================================
total_pres = presence.sum().item()

print(f"\n💾 Saving to {output_path}...")
print(f"   Presence matrix  : {list(presence.shape)}  "
      f"({presence.nbytes / 1e6:.1f} MB)")
print(f"   Bio-opp matrix   : {list(bio_opp_mat.shape)}  "
      f"({bio_opp_mat.nbytes / 1e6:.1f} MB)")
print(f"   Total presences  : {total_pres:,}  "
      f"(+{total_pres - n_seeded:,} over SINAS alone)")
print(f"   Species fetched  : "
      f"{len(completed) + len(new_completed):,} / {len(resolvable):,}")
if all_failed:
    print(f"   Permanently failed: {len(all_failed):,} species")

torch.save({
    'presence':           presence,         # (S, C) bool
    'bio_opp_mat':        bio_opp_mat,      # (S, C) float16
    'species_order':      all_species_ids,  # list[int] gbif_ids — same order as sp_to_idx
    'countries_order':    all_country_names,
    'complete':           len(all_failed) == 0,
    'permanently_failed': all_failed,
}, output_path)

print(f"✅ Saved.")

# Remove checkpoint only if fully complete with no failures
if len(all_failed) == 0 and checkpoint_path.exists():
    checkpoint_path.unlink()
    print("   Checkpoint removed.")
elif all_failed:
    print(f"   Checkpoint retained — re-run script to retry "
          f"{len(all_failed):,} permanently failed species.")


💾 Saving to c:\Users\simon\Documents\GitHub\horizon-scanner\data\gbif_presence.pt...
   Presence matrix  : [167357, 289]  (48.4 MB)
   Bio-opp matrix   : [167357, 289]  (96.7 MB)
   Total presences  : 3,311,938  (+2,955,432 over SINAS alone)
   Species fetched  : 167,454 / 167,357
✅ Saved.
   Checkpoint removed.
